# Predicting day of year from local weather

Compare air temperature alone (`tas`) with air temperature and dew point
(`tas + tdew`). For each input set and lag, fit the same pooled ridge probe
to raw weather histories and to frozen features from the matching weather-only
mortality model. The probe predicts sine and cosine of the annual phase
from district-level weather; mortality weights remain frozen.

The error summaries use the saved validation dates. The Cartesian and polar
plots pool training and validation dates across years for windows of 1, 6, 10,
20, and 30 days. Both use the same 366-day calendar, with February 29
reserved for leap years. Member 000 is used by default.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

cwd = Path.cwd()
repo = next(
    (
        path for path in (cwd, *cwd.parents)
        if (path / "configs" / "training.example.yaml").is_file()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError("Open the notebook from the sari-weather-mortality checkout.")

sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load helpers from this checkout even if Jupyter cached another notebooks package.
utils_path = repo / "notebooks" / "utils.py"
spec = spec_from_file_location("sari_notebook_utils", utils_path)
notebook_utils = module_from_spec(spec)
spec.loader.exec_module(notebook_utils)


from calendar import month_abbr

from matplotlib.colors import PowerNorm
from matplotlib.ticker import PercentFormatter
import torch

from sari_t.data.weather import load_weather
from sari_t.run.checkpoint import load_checkpoint, load_run_config, restore_predictor

## Load matching weather-only checkpoints

In [ ]:
members = (0,)
inspection_window = 9
groups = {
    "tas": ("temperature_kernel_weather_only", ("tas",)),
    "tas + tdew": ("temperature_kernel_weather_only_tas_tdew", ("tas", "tdew")),
}
runs = {}
for feature, (group, channels) in groups.items():
    feature_runs = {}
    for entry in sorted((repo / "runtime" / "training" / "sets" / group).iterdir()):
        run = entry.resolve()
        saved = load_run_config(run / "config.yaml")
        lag = saved.model_config.kernel_T_days
        if (lag in feature_runs or saved.selected_weather != channels
                or saved.model_config.include_sari or saved.model_config.include_covid):
            raise ValueError(f"Unexpected weather-only configuration in {run}")
        feature_runs[lag] = run
    if set(feature_runs) != set(range(1, 31)):
        raise ValueError(f"Incomplete lag sweep for {feature}")
    runs[feature] = feature_runs

models = {
    feature: restore_predictor(load_checkpoint(
        feature_runs[inspection_window] / "checkpoints" / f"final_{members[0]:03d}.pt"
    ), device="cpu").eval()
    for feature, feature_runs in runs.items()
}
model = models["tas + tdew"]
model

## Fit and score the probes

Run the fitting cells once. Both plot types reuse the counts in memory.

In [ ]:
config = load_run_config(runs["tas"][inspection_window] / "config.yaml")
period = config.period
dates = pd.date_range(period.start_date, period.end_date)
weeks = len(dates) // 7
fraction = config.training_config.validation_fraction
validation_days = 7 * int(weeks * fraction)
train_days = len(dates) - validation_days
if len(dates) % 7 or not 0 < validation_days < len(dates):
    raise ValueError("Expected complete training and validation weeks.")
validation_start = dates[train_days]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
year_lengths = np.where(dates.is_leap_year, 366, 365)
day_of_year = dates.dayofyear.to_numpy() - 1
phase = 2 * np.pi * day_of_year / year_lengths
target = torch.as_tensor(np.column_stack((np.cos(phase), np.sin(phase))),
                         device=device, dtype=torch.float64)
train = torch.as_tensor(dates < validation_start, device=device)
test = ~train
days_in_year = torch.as_tensor(year_lengths, device=device)
calendar_columns = torch.as_tensor(day_of_year + ((year_lengths == 365) & (day_of_year >= 59)),
                                  device=device)
print(f"Training {period.start_date} to {dates[train_days - 1].date()}")
print(f"Validation {validation_start.date()} to {period.end_date}")

In [ ]:
def pooled_probe(batches, *, collect_heatmap=False):
    """Fit one standardized ridge decoder with bounded memory, then score each district."""
    target_fit = target[train]
    intercept = target_fit.mean(dim=0)
    centered_target = target_fit - intercept
    count = 0
    for batch in batches():
        batch = batch.to(device=device, dtype=target.dtype)
        fit = batch[:, train].reshape(batch.shape[0] * train_days, -1)
        if count == 0:
            width = fit.shape[1]
            feature_sum = fit.new_zeros(width)
            cross_features = fit.new_zeros((width, width))
            cross_target = fit.new_zeros((width, 2))
        feature_sum += fit.sum(dim=0)
        cross_features += fit.T @ fit
        cross_target += fit.T @ centered_target.repeat(batch.shape[0], 1)
        count += len(fit)

    center = feature_sum / count
    covariance = cross_features - count * torch.outer(center, center)
    scale = (covariance.diag() / count).clamp_min(0).sqrt()
    scale = torch.where(scale == 0, 1.0, scale)
    standardized = covariance / torch.outer(scale, scale)
    penalty = 0.01 * count * torch.eye(width, device=device, dtype=target.dtype)
    weights = torch.linalg.solve(standardized + penalty, cross_target / scale[:, None])
    actual = torch.atan2(target[test, 1], target[test, 0])
    errors = []
    daily_errors = []
    if collect_heatmap:
        counts = torch.zeros(366 * 366, device=device, dtype=torch.long)

    for batch in batches():
        batch = batch.to(device=device, dtype=target.dtype)
        held = batch[:, test].reshape(batch.shape[0], validation_days, width)
        predicted = ((held - center) / scale) @ weights + intercept
        angle = torch.atan2(predicted[..., 1], predicted[..., 0])
        difference = (angle - actual + torch.pi) % (2 * torch.pi) - torch.pi
        if collect_heatmap:
            all_days = batch.reshape(batch.shape[0], len(dates), width)
            all_predictions = ((all_days - center) / scale) @ weights + intercept
            all_angles = torch.atan2(all_predictions[..., 1], all_predictions[..., 0])
            phase_fraction = all_angles.remainder(2 * torch.pi) / (2 * torch.pi)
            estimated = (phase_fraction * days_in_year).round().long() % days_in_year
            estimated = estimated + ((days_in_year == 365) & (estimated >= 59))
            indices = estimated * 366 + calendar_columns
            counts += torch.bincount(indices.flatten(), minlength=counts.numel())
        absolute_days = difference.abs() * 365.25 / (2 * torch.pi)
        errors.extend(torch.quantile(absolute_days, 0.5, dim=1).cpu().tolist())
        daily_errors.append(absolute_days)
    daily_errors = torch.cat(daily_errors)
    seasonal = {}
    for season, mask in {**season_masks, **month_masks}.items():
        values = daily_errors[:, mask].flatten()
        if values.numel():
            q1, median, q3 = torch.quantile(values, values.new_tensor([0.25, 0.5, 0.75])).cpu().tolist()
            seasonal[season] = (values.numel(), median, q1, q3, q3 - q1)
    heatmap = counts.reshape(366, 366).cpu() if collect_heatmap else None
    return errors, seasonal, heatmap

In [ ]:
validation_months = dates[dates >= validation_start].month
season_masks = {
    season: torch.as_tensor(validation_months.isin(months), device=device)
    for season, months in {
        "Winter": (12, 1, 2), "Spring": (3, 4, 5),
        "Summer": (6, 7, 8), "Autumn": (9, 10, 11),
    }.items()
}

month_masks = {
    month_abbr[month]: torch.as_tensor(validation_months == month, device=device)
    for month in range(1, 13)
}

heatmap_lags = (1, 6, 10, 20, 30)
heatmaps = {}
rows = []
seasonal_rows = []
for family, family_runs in runs.items():
    print(f"Fitting probes for {family}", flush=True)
    first = load_run_config(next(iter(family_runs.values())) / "config.yaml")
    weather = load_weather(first.data_config, first.selected_weather)
    first_day = (period.start_date - weather.start_date).days
    end = first_day + len(dates)
    if end > weather.values.shape[-1] or first_day < max(family_runs) - 1:
        raise ValueError(f"Missing weather dates or lag history for {family}")
    for lag, run in sorted(family_runs.items()):
        saved = load_run_config(run / "config.yaml")
        if (saved.period != period or saved.training_config.validation_fraction != fraction
                or saved.selected_weather != first.selected_weather or saved.data_config != first.data_config):
            raise ValueError(f"Inconsistent dates, split, or data in {run}")
        start = first_day - lag + 1
        window = weather.values[..., start:end]
        def raw_batches():
            for batch in window.split(16):
                yield batch.unfold(-1, lag, 1).transpose(1, 2)

        raw_errors, raw_seasons, raw_heatmap = pooled_probe(
            raw_batches, collect_heatmap=lag in heatmap_lags
        )
        if raw_heatmap is not None:
            heatmaps[family, "Raw history", lag] = raw_heatmap
        for member in members:
            checkpoint = run / "checkpoints" / f"final_{member:03d}.pt"
            model = restore_predictor(load_checkpoint(checkpoint), device=device).eval().effect_model

            def feature_batches():
                for batch in window.split(16):
                    with torch.inference_mode():
                        features = model.conv1(model.batch_norm(batch.to(device))).exp()
                    yield features.transpose(1, 2)

            frozen_errors, frozen_seasons, frozen_heatmap = pooled_probe(
                feature_batches, collect_heatmap=lag in heatmap_lags
            )
            if frozen_heatmap is not None:
                key = (family, "Frozen features", lag)
                if key in heatmaps:
                    heatmaps[key] += frozen_heatmap
                else:
                    heatmaps[key] = frozen_heatmap
            for feature, seasons in (
                ("Raw history", raw_seasons),
                ("Frozen features", frozen_seasons),
            ):
                for season, stats in seasons.items():
                    seasonal_rows.append((family, lag, member, feature, season, *stats))
            for district, error in enumerate(frozen_errors):
                rows.append((family, lag, member, district, raw_errors[district], error))

## Validation errors

In [ ]:
metrics = ["Raw history", "Frozen features"]
results = pd.DataFrame(rows, columns=["Model", "Lag days", "Member", "District index", *metrics])
district_errors = results.groupby(["Model", "Lag days", "District index"])[metrics].median()
summary = district_errors.groupby(["Model", "Lag days"])[metrics].median()
display(summary.round(2))
seasonal_errors = pd.DataFrame(seasonal_rows, columns=[
    "Model", "Lag days", "Member", "Features", "Season", "District-days",
    "Median error (days)", "Q1 (days)", "Q3 (days)", "IQR (days)",
]).set_index(["Model", "Lag days", "Member", "Features", "Season"])
monthly_errors = seasonal_errors[seasonal_errors.index.get_level_values("Season").isin(month_masks)]
monthly_errors = monthly_errors.rename_axis(index={"Season": "Month"})
seasonal_errors = seasonal_errors[seasonal_errors.index.get_level_values("Season").isin(season_masks)]
display(seasonal_errors.round(2))

curves = {
    ("tas", "Raw history"): ("Raw tas", "#0072B2", "--"),
    ("tas + tdew", "Raw history"): ("Raw tas + tdew", "#D55E00", "--"),
    ("tas", "Frozen features"): ("Frozen tas", "#0072B2", "-"),
    ("tas + tdew", "Frozen features"): ("Frozen tas + tdew", "#D55E00", "-"),
}

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for (family, feature), (label, color, style) in curves.items():
    values = summary.loc[family, feature]
    ax.plot(values.index, values, label=label, color=color, linestyle=style)
ax.set(title="Overall validation", xlabel="Lag days", ylabel="Median district date error (days)")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
for season in season_masks:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for (family, feature), (label, color, style) in curves.items():
        subset = seasonal_errors.xs((family, feature, season), level=("Model", "Features", "Season"))
        values = subset.groupby(level="Lag days").median(numeric_only=True)
        ax.plot(values.index, values["Median error (days)"], color=color, linestyle=style, label=label)
        ax.fill_between(values.index, values["Q1 (days)"], values["Q3 (days)"], color=color, alpha=0.10)
    ax.set(title=f"{season} validation", xlabel="Lag days", ylabel="Median date error (days)")
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

fig, axes = plt.subplots(4, 3, figsize=(13, 11), sharex=True, sharey=True, layout="constrained")
for month, ax in zip(month_masks, axes.flat):
    for (family, feature), (label, color, style) in curves.items():
        subset = monthly_errors.xs((family, feature, month), level=("Model", "Features", "Month"))
        values = subset.groupby(level="Lag days").median(numeric_only=True)
        ax.plot(values.index, values["Median error (days)"], color=color, linestyle=style, label=label)
        ax.fill_between(values.index, values["Q1 (days)"], values["Q3 (days)"], color=color, alpha=0.10)
    ax.set_title(month)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncol=4, frameon=False)
fig.supxlabel("Lag days")
fig.supylabel("Median validation date error (days)")
plt.show()

## Predicted versus true day of year

The diagonal indicates correct predictions. Columns are normalized to shares of district predictions.

In [ ]:
shares = {}
for key, counts in heatmaps.items():
    shares[key] = (counts / counts.sum(dim=0).clamp_min(1)).numpy()
positive = np.concatenate([values[values > 0] for values in shares.values()])
norm = PowerNorm(gamma=0.5, vmin=0, vmax=np.quantile(positive, 0.995))
day_edges = np.arange(367) + 0.5
fig, axes = plt.subplots(len(heatmap_lags), len(curves), figsize=(15, 16),
                         sharex=True, sharey=True, layout="constrained")
for row, lag in enumerate(heatmap_lags):
    for col, ((family, feature), (label, _, _)) in enumerate(curves.items()):
        ax = axes[row, col]
        mesh = ax.pcolormesh(day_edges, day_edges, shares[family, feature, lag],
                             cmap="Blues", norm=norm)
        ax.plot([1, 366], [1, 366], color="#D55E00", linewidth=0.9)
        if row == 0:
            ax.set_title(label)
        if col == 0:
            ax.set_ylabel(f"{lag}-day window\nPredicted day")
for ax in axes[-1]:
    ax.set_xlabel("True day of year")
fig.colorbar(mesh, ax=axes, extend="max", format=PercentFormatter(1),
             label="Share of district predictions")
plt.show()

## Circular date errors

Angle shows the true day of year clockwise from January 1. The orange ring
marks zero error; inward estimates are early and outward estimates are late.
Errors wrap at half a year. The colour scale matches the Cartesian plot.

In [ ]:
year_days = 366
half_year = year_days // 2
true_days = np.arange(year_days)
signed_days = np.arange(-half_year, half_year)
estimated_rows = (signed_days[:, None] + true_days[None, :]) % year_days
angle_edges = (np.arange(year_days + 1) - 0.5) * 2 * np.pi / year_days
radius_edges = np.arange(year_days + 1)
ring_angles = np.linspace(0, 2 * np.pi, 367)
month_starts = pd.date_range("2000-01-01", periods=12, freq="MS")[::3]
tick_days = np.array([-120, -60, 0, 60, 120, 182])
fig, axes = plt.subplots(len(heatmap_lags), len(curves), figsize=(15, 16),
                         subplot_kw={"projection": "polar"}, layout="constrained")
for row, lag in enumerate(heatmap_lags):
    for col, ((family, feature), (label, _, _)) in enumerate(curves.items()):
        ax = axes[row, col]
        ax.grid(False)
        values = shares[family, feature, lag][estimated_rows, true_days]
        mesh = ax.pcolormesh(angle_edges, radius_edges, values, cmap="Blues", norm=norm)
        ax.plot(ring_angles, np.full_like(ring_angles, half_year + 0.5),
                color="#D55E00", linewidth=1)
        ax.set_theta_zero_location("N")
        ax.set_theta_direction(-1)
        ax.set_xticks((month_starts.dayofyear - 1) * 2 * np.pi / year_days,
                      month_starts.strftime("%b"))
        ax.set_rticks(tick_days + half_year + 0.5,
                      [f"{day:+d}d" if day else "0d" for day in tick_days])
        ax.set_rlabel_position(40)
        ax.set_ylim(0, year_days)
        ax.grid(alpha=0.2)
        if row == 0:
            ax.set_title(label, pad=16)
        if col == 0:
            ax.set_ylabel(f"{lag}-day window", labelpad=26)
fig.colorbar(mesh, ax=axes, extend="max", format=PercentFormatter(1),
             label="Share of district predictions")
plt.show()